test notebook. 
attempting to visualize urdf


In [4]:
# Native urdfpy: use URDF next to meshes with relative paths (see urdfpy URDF.load docstring).
from pathlib import Path
import numpy as np
from urdfpy import URDF

root = Path(r"/home/sybbure/Desktop/REWIND_SENIOR_DESIGN/")
urdf_path = root / "glove_sim/outputs/aligned/urdf_frames/rewind_glove_000300.urdf"
robot = URDF.load(str(urdf_path))

# If you must load the OnShape URDF with package://mesh URIs instead, use:
#   sys.path.insert(0, str(root / "glove_sim"))
#   from src.urdfpy_vis import load_robot
#   robot = load_robot(root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf", root / "rewind_glove_assembly/meshes")

# 2. Visualize the glove in its rest state
print("Visualizing rest pose. Close the window to continue...")
robot.show()

AttributeError: module 'numpy' has no attribute 'float'.
`np.float` was a deprecated alias for the builtin `float`. To avoid this error in existing code, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [2]:
import trimesh

# 1. Get the forward kinematics for all visual meshes
# This returns a dictionary mapping trimesh objects to their 4x4 pose matrices
fk = robot.visual_trimesh_fk()

# 2. Create a list to hold the transformed meshes
meshes = []
for mesh, pose in fk.items():
    # We must copy and apply the transform so the parts are in the right place
    m = mesh.copy()
    m.apply_transform(pose)
    meshes.append(m)

# 3. Create a trimesh Scene from the list of meshes
scene = trimesh.Scene(meshes)

# 4. Export the scene to a GLB file
scene.export('my_robot.glb')

b'glTF\x02\x00\x00\x00\xc0-\n\x00\x90<\x00\x00JSON{"scene":0,"scenes":[{"nodes":[0]}],"asset":{"version":"2.0","generator":"https://github.com/mikedh/trimesh"},"accessors":[{"componentType":5125,"type":"SCALAR","bufferView":0,"count":17532,"max":[2903],"min":[0]},{"componentType":5126,"type":"VEC3","byteOffset":0,"bufferView":1,"count":2904,"max":[-0.07437600195407867,0.06638380140066147,-0.05108169838786125],"min":[-0.1755754053592682,-0.01851559989154339,-0.07408169656991959]},{"componentType":5125,"type":"SCALAR","bufferView":2,"count":3054,"max":[508],"min":[0]},{"componentType":5126,"type":"VEC3","byteOffset":0,"bufferView":3,"count":509,"max":[-0.1408682018518448,-0.0005890000029467046,-0.05100160092115402],"min":[-0.1735534965991974,-0.030536500737071037,-0.07404647767543793]},{"componentType":5125,"type":"SCALAR","bufferView":4,"count":2424,"max":[403],"min":[0]},{"componentType":5126,"type":"VEC3","byteOffset":0,"bufferView":5,"count":404,"max":[-0.14728578925132751,-0.0122784

### Why the raw OnShape / ROS URDF broke `URDF.load`

1. **`package://pkg/meshes/file.stl`** — ROS resolves that via package search paths. **urdfpy does not**; it uses `os.path.join(urdf_directory, filename)` for non-absolute paths, so you get a bogus path that still contains `package://…`.
2. **What upstream expects** — Same as `urdfpy.utils.get_filename`: mesh `filename` should be **relative to the `.urdf` file** (or an absolute path). The companion `rewind_glove_for_urdfpy.urdf` uses `../meshes/...` for that reason.
3. **`load_robot`** — Rewrites `package://` to a path **relative to the URDF folder** (using your `mesh_dir` to compute `relpath`), strips invalid empty `<texture/>` tags, writes a temp `.urdf` **next to** the real one, then calls **`URDF.load`** — same parser path as “native” urdfpy.

### `glove_sim` / MuJoCo

MuJoCo builds MJCF from the same URDF mesh attributes: it joins **`cfg.MESH_DIR`** with the attribute (after stripping `package://` to a basename, or normalizing `../meshes/...`). Use **binary STLs** under `glove_sim/assets/meshes` (`convert_meshes_for_mujoco.py`) so MuJoCo’s loader is happy; urdfpy can still load from `rewind_glove_assembly/meshes` (trimesh accepts ASCII STL).

In [3]:
# load_robot: same urdfpy URDF.load path after fixing package:// + textures
import sys
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

from src.urdfpy_vis import load_robot

canonical_urdf = root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf"
assembly_mesh_dir = root / "rewind_glove_assembly/meshes"

robot_via_helper = load_robot(canonical_urdf, assembly_mesh_dir)
print("load_robot(canonical URDF, assembly/meshes):", len(robot_via_helper.links), "links")
print("  actuated joints:", len(robot_via_helper.actuated_joints))

assert len(robot_via_helper.links) == len(robot.links), "should match native `robot` from cell 1"
assert {j.name for j in robot_via_helper.actuated_joints} == {j.name for j in robot.actuated_joints}
print("OK: link count and actuated joint names match native urdfpy load.")

ModuleNotFoundError: No module named 'src'

In [4]:
# Resolve one mesh the same way urdfpy vs MuJoCo pipeline do
import os
import xml.etree.ElementTree as ET
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")


def resolve_like_urdfpy(urdf_file: Path, mesh_filename_attr: str) -> Path:
    """urdfpy.utils.get_filename(urdf_dir, file_path)."""
    base = str(urdf_file.parent.resolve())
    fn = mesh_filename_attr
    if os.path.isabs(fn):
        return Path(fn)
    return Path(os.path.normpath(os.path.join(base, fn)))


def resolve_like_mujoco_glove_ik(mesh_dir: Path, mesh_filename_attr: str) -> Path:
    """glove_ik: strip package:// to basename chain, else keep attr; normpath(mesh_dir / fn)."""
    fn = mesh_filename_attr
    if fn.startswith("package://"):
        fn = fn.split("/", 2)[-1]
        fn = fn.split("/", 1)[-1]
        fn = fn.split("/", 1)[-1]
    return Path(os.path.normpath(str(mesh_dir.resolve() / fn)))

urdf_for_urdfpy = root / "rewind_glove_assembly/urdf/rewind_glove_for_urdfpy.urdf"
mesh_attr = next(m.get("filename") for m in ET.parse(urdf_for_urdfpy).getroot().iter("mesh"))

p_urdfpy = resolve_like_urdfpy(urdf_for_urdfpy, mesh_attr)
mujoco_mesh_dir = root / "glove_sim/assets/meshes"
p_mujoco = resolve_like_mujoco_glove_ik(mujoco_mesh_dir, mesh_attr)

print("Example <mesh filename=...>:", repr(mesh_attr))
print("  urdfpy (join URDF dir + attribute):", p_urdfpy, "| exists:", p_urdfpy.is_file())
print("  MuJoCo cfg.MESH_DIR join + normpath:", p_mujoco, "| exists:", p_mujoco.is_file())
if p_urdfpy.is_file() and p_mujoco.is_file():
    print("OK: visualization (assembly) and MuJoCo (binary copy) both find this STL.")
elif p_urdfpy.is_file() and not p_mujoco.is_file():
    print("Run from repo root: python glove_sim/convert_meshes_for_mujoco.py")

Example <mesh filename=...>: '../meshes/Hand Mount.stl'
  urdfpy (join URDF dir + attribute): C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\rewind_glove_assembly\meshes\Hand Mount.stl | exists: True
  MuJoCo cfg.MESH_DIR join + normpath: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\assets\meshes\Hand Mount.stl | exists: True
OK: visualization (assembly) and MuJoCo (binary copy) both find this STL.


In [5]:
# MuJoCo: same URDF + cfg.MESH_DIR as pipeline / diagnostic
import sys
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

import config as cfg
from src.glove_ik import GloveSimulator

print("URDF:", cfg.URDF_PATH)
print("MESH_DIR:", cfg.MESH_DIR)
try:
    sim = GloveSimulator(cfg.URDF_PATH, cfg.MESH_DIR)
    print("MuJoCo GloveSimulator OK — nq=", sim.model.nq, "nv=", sim.model.nv, "nbody=", sim.model.nbody)
except Exception as e:
    print("MuJoCo load failed:", type(e).__name__, e)
    print("If missing/corrupt STLs under assets/meshes, run: python glove_sim/convert_meshes_for_mujoco.py")

URDF: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\rewind_glove_assembly\urdf\rewind_glove_assembly.urdf
MESH_DIR: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\assets\meshes
MuJoCo GloveSimulator OK — nq= 16 nv= 15 nbody= 22


### Visualize via updated `urdfpy_vis.get_glove_scene`

This uses the updated `load_robot`/`get_glove_scene` path (same FK path as urdfpy, mesh path fixes included) and exports GLBs you can open directly.

In [6]:
# Export rest + bent scenes from updated urdfpy_vis
import sys
from pathlib import Path
import numpy as np

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
sys.path.insert(0, str(root / "glove_sim"))

from src.urdfpy_vis import load_robot, get_glove_scene

canonical_urdf = root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf"
assembly_mesh_dir = root / "rewind_glove_assembly/meshes"
robot_vis = load_robot(canonical_urdf, assembly_mesh_dir)

# IMPORTANT: get_glove_scene expects HAND-MOUNT world pose, not URDF-root pose.
# Using identity here detaches parts visually (wrong frame). For URDF-root at identity,
# hand_mount must be translated by ROOT_TO_HANDMOUNT.
T_hand_mount_world = np.eye(4, dtype=float)
T_hand_mount_world[:3, 3] = np.array([-0.157876, 0.0663838, -0.0660817], dtype=float)

# Build joint configs using all actuated joints, then override two for a visible bend
joint_cfg_rest = {j.name: 0.0 for j in robot_vis.actuated_joints}
joint_cfg_bent = dict(joint_cfg_rest)
joint_cfg_bent["revolute_3_0"] = np.deg2rad(45.0)
joint_cfg_bent["revolute_9_0"] = np.deg2rad(45.0)

scene_rest = get_glove_scene(robot_vis, joint_cfg_rest, T_hand_mount_world)
scene_bent = get_glove_scene(robot_vis, joint_cfg_bent, T_hand_mount_world)

out_dir = root / "glove_sim_v2/outputs/urdfpy_vis_scene"
out_dir.mkdir(parents=True, exist_ok=True)
out_rest = out_dir / "glove_rest_urdfpy_vis.glb"
out_bent = out_dir / "glove_bent_urdfpy_vis.glb"

scene_rest.export(str(out_rest))
scene_bent.export(str(out_bent))

print("Exported:")
print(" ", out_rest)
print(" ", out_bent)
print("Tip: open these in f3d/Blender to confirm links stay connected through joints.")

Exported:
  C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\glove_rest_urdfpy_vis.glb
  C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\glove_bent_urdfpy_vis.glb
Tip: open these in f3d/Blender to confirm links stay connected through joints.


In [7]:
# Optional: short sweep to verify continuity of linkage motion
sweep_dir = out_dir / "index_tip_sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

for deg in range(0, 91, 15):
    cfg = dict(joint_cfg_rest)
    cfg["revolute_9_0"] = np.deg2rad(float(deg))
    scene = get_glove_scene(robot_vis, cfg, T_hand_mount_world)
    scene.export(str(sweep_dir / f"index_{deg:03d}.glb"))

print(f"Exported sweep frames to: {sweep_dir}")

Exported sweep frames to: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\index_tip_sweep


In [8]:
# Verify exported GLBs: custom vs native-side reference built from visual_trimesh_fk
import os
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import trimesh
from urdfpy import URDF

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
assembly_urdf = root / "rewind_glove_assembly/urdf/rewind_glove_assembly.urdf"
assembly_mesh_dir = root / "rewind_glove_assembly/meshes"
urdf_dir = assembly_urdf.parent.resolve()

# Build native oracle robot from canonical URDF with minimal sanitation
tr = ET.parse(str(assembly_urdf))
for m in tr.getroot().iter("mesh"):
    fn = m.get("filename", "")
    if fn.startswith("package://"):
        base = fn.split("/", 2)[-1].split("/", 1)[-1].split("/", 1)[-1]
        rel = os.path.relpath(assembly_mesh_dir / base, start=urdf_dir).replace("\\", "/")
        m.set("filename", rel)
for material in tr.getroot().iter("material"):
    for tex in list(material.findall("texture")):
        if not tex.attrib or not tex.get("filename"):
            material.remove(tex)
fd, tmp_path = tempfile.mkstemp(prefix="._native_cmp_", suffix=".urdf", dir=str(urdf_dir))
os.close(fd)
tr.write(tmp_path, encoding="utf-8", xml_declaration=True)
robot_native = URDF.load(tmp_path)
os.unlink(tmp_path)

# Build native-side GLB in same frame conversion as get_glove_scene
cfg = dict(joint_cfg_rest)
ydown_to_yup = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, -1, 0, 0], [0, 0, 0, 1]], dtype=float)
scene_native = trimesh.Scene()
for mesh, T in robot_native.visual_trimesh_fk(cfg=cfg).items():
    m = mesh.copy()
    m.apply_transform(ydown_to_yup @ T)
    scene_native.add_geometry(m)

cmp_dir = root / "glove_sim_v2/outputs/parity_check"
cmp_dir.mkdir(parents=True, exist_ok=True)
native_ref_path = cmp_dir / "native_rest.glb"
custom_path = out_rest
scene_native.export(str(native_ref_path))

s_native = trimesh.load(str(native_ref_path), force="scene")
s_custom = trimesh.load(str(custom_path), force="scene")

print("native_ref:", native_ref_path)
print("custom:", custom_path)
print("native bounds:\n", s_native.bounds)
print("custom bounds:\n", s_custom.bounds)
print("max |bounds delta|:", float(np.max(np.abs(s_native.bounds - s_custom.bounds))))
print("n geom native/custom:", len(s_native.geometry), len(s_custom.geometry))

native_ref: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\parity_check\native_rest.glb
custom: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim_v2\outputs\urdfpy_vis_scene\glove_rest_urdfpy_vis.glb
native bounds:
 [[-0.26487464 -0.19585221 -0.23269069]
 [-0.074376   -0.01808163  0.063357  ]]
custom bounds:
 [[-0.26487464 -0.19585221 -0.23269069]
 [-0.074376   -0.01808163  0.063357  ]]
max |bounds delta|: 0.0
n geom native/custom: 21 21


In [9]:
# Run authoritative parity export/compare and print where GLBs are written
import json
import subprocess
from pathlib import Path

root = Path(r"C:/Users/dnxjc/Desktop/CURR/PROJECTS/rewind")
cmd = ["python", "glove_sim/compare_visual_exports.py"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError(f"Parity script failed with exit code {res.returncode}")

report_path = root / "glove_sim/outputs/parity_exports/report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

print("Authoritative report:", report_path)
print("\nScripts responsible:")
print("  Ground truth/native export: glove_sim/compare_visual_exports.py (_prepare_native_oracle_from_canonical + _scene_from_native)")
print("  Reconstructed/in-house export: glove_sim/compare_visual_exports.py (calls glove_sim/src/urdfpy_vis.py:get_glove_scene)")
print("  In-house loader/path handling: glove_sim/src/urdfpy_vis.py (load_robot)")

print("\nPer-config GLB outputs:")
for r in report["results"]:
    print(f"  {r['config']}")
    print(f"    native : {r['native_glb']}")
    print(f"    custom : {r['custom_glb']}")

# Quick convenience aliases for rest pose files
rest_native = root / "glove_sim/outputs/parity_exports/rest/native.glb"
rest_custom = root / "glove_sim/outputs/parity_exports/rest/custom.glb"
print("\nRest pose quick paths:")
print("  native:", rest_native)
print("  custom:", rest_custom)

Running: python glove_sim/compare_visual_exports.py
PASS: rest pose and small sweep parity checks are green.
Report: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\outputs\parity_exports\report.json

Authoritative report: C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\outputs\parity_exports\report.json

Scripts responsible:
  Ground truth/native export: glove_sim/compare_visual_exports.py (_prepare_native_oracle_from_canonical + _scene_from_native)
  Reconstructed/in-house export: glove_sim/compare_visual_exports.py (calls glove_sim/src/urdfpy_vis.py:get_glove_scene)
  In-house loader/path handling: glove_sim/src/urdfpy_vis.py (load_robot)

Per-config GLB outputs:
  rest
    native : C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\outputs\parity_exports\rest\native.glb
    custom : C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\outputs\parity_exports\rest\custom.glb
  thumb45
    native : C:\Users\dnxjc\Desktop\CURR\PROJECTS\rewind\glove_sim\outputs\parity_ex